# TOKENIZER


## Imports

In [ ]:

import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import re


## CREATING TOKENS


In [ ]:
with open("../data/raw/the-verdict.txt", "r", encoding="utf-8") as f:
    raw = f.read()


print(len(raw))
print(raw[:99])

In [ ]:

text_test = "Hello, world! This is a test of the text processing system."
result = re.split(r'([,.]|\s)', text_test) # split on commas, periods, or spaces
print(result)

# remove empty strings(whitespaces)
result = [x for x in result if x.strip()]
print(result)



<div class="alert alert-block alert-success">

REMOVING WHITESPACES OR NOT


When developing a simple tokenizer, whether we should encode whitespaces as
separate characters or just remove them depends on our application and its
requirements. Removing whitespaces reduces the memory and computing
requirements. However, keeping whitespaces can be useful if we train models that
are sensitive to the exact structure of the text (for example, Python code, which is
sensitive to indentation and spacing). Here, we remove whitespaces for simplicity
and brevity of the tokenized outputs. Later, we will switch to a tokenization scheme
that includes whitespaces.

</div>

In [ ]:
# This is our basic tokenization pattern for English, can vaiate depending on the language, ex: Chinese, Python, etc. and needs
test_text = "Hello, world! This is a test-- ,of the text processing system?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', test_text)
result = [x for x in result if x.strip()]
print(result)

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw)
preprocessed = [x for x in preprocessed if x.strip()]
print(preprocessed[:30])
print(len(preprocessed))

## CREATE TOKEN ID'S

#### Let's now create a list of all unique tokens and sort them alphabetically to determine the vocabulary size


In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

In [ ]:
vocab = {word:i for i, word in enumerate(all_words)}
print(vocab["the"])

## CREATE TOKENIZER CLASS

In [ ]:
class SimpleTokenizerV1: 
    def __init__(self, vocab): 

        self.str_to_int = vocab
        self.int_to_str = {v:k for k, v in vocab.items()}

    def encode(self, text): 
        preprocessed_text = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [x for x in preprocessed_text if x.strip()]

        ids = [self.str_to_int[x] for x in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[x] for x in ids])
        # remove spaces before punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that,"
ids = tokenizer.encode(text)
print(ids)


In [ ]:
decoded_text = tokenizer.decode(ids)
print(decoded_text)

## SPECIAL CONTEXT TOKENS

#### What happen if a words passed to encode is not present in the vocabulary ?

In [ ]:
# text = "Hello, world! This is a test of the text processing system."
# ids = tokenizer.encode(text)
# print(ids)

#### In this case we will add 2 special tokens "<|endoftext|>", "<|unk|>"

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
print(tokens:=tokenizer.encode(text))
print(tokenizer.decode(tokens))

<div class="alert alert-block alert-warning">

So far, we have discussed tokenization as an essential step in processing text as input to
LLMs. Depending on the LLM, some researchers also consider additional special tokens such
as the following:

[BOS] (beginning of sequence): This token marks the start of a text. It
signifies to the LLM where a piece of content begins.

[EOS] (end of sequence): This token is positioned at the end of a text,
and is especially useful when concatenating multiple unrelated texts,
similar to <|endoftext|>. For instance, when combining two different
Wikipedia articles or books, the [EOS] token indicates where one article
ends and the next one begins.

[PAD] (padding): When training LLMs with batch sizes larger than one,
the batch might contain texts of varying lengths. To ensure all texts have
the same length, the shorter texts are extended or "padded" using the
[PAD] token, up to the length of the longest text in the batch.

</div>


<div class="alert alert-block alert-warning">

Note that the tokenizer used for GPT models does not need any of these tokens mentioned
above but only uses an <|endoftext|> token for simplicity

</div>

<div class="alert alert-block alert-warning">

the tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding tokenizer, which breaks
down words into subword units
</div>

## BYTE PAIR ENCODING (BPE)


#### BPE tokenizer are quite complexe and time comsuming we will use the OpenAI tokeniser used in GPT2, GPT3, ... 

In [ ]:
%pip install tiktoken

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

## CREATE INPUT-TARGET PAIR

In [ ]:
with open("../data/raw/the-verdict.txt", "r", encoding="utf-8") as f:
    raw = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw)
len(enc_text) # numbers of tokens (BPE) in the text


In [ ]:
context_size = 4 # number of tokens to consider before predicting the next token
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens) 
#to predict the next word in the sequence. 
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_text[:context_size]
y = enc_text[1:context_size + 1]

print(x)
print(y)



In [ ]:
for i in range(1, context_size + 1): 
    context = enc_text[:i]
    desired = enc_text[i]
    print(context, "->", desired)

## IMPLEMENTING DATA LOADER

In [ ]:
%pip install torch

In [ ]:


class GPTDatasetV1(Dataset): 
    def __init__(self, txt, tokenizer, max_lenght, stride):
        self.input_ids = []
        self.target_ids = []

        #Tokenizer entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_lenght, stride):
            input_chunks = token_ids[i:i+max_lenght]
            target_chunks = token_ids[i+1:i+max_lenght+1]

            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

In [ ]:
with open("../data/raw/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:

print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

<div class="alert alert-block alert-warning">

The first_batch variable contains two tensors: the first tensor stores the input token IDs,
and the second tensor stores the target token IDs. 

Since the max_length is set to 4, each of the two tensors contains 4 token IDs. 

Note that an input size of 4 is relatively small and only chosen for illustration purposes. It is common to train LLMs with input sizes of at least
256.
    
</div>

In [ ]:
# Effect of batch_size
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

<div class="alert alert-block alert-info">
    
Note that we increase the stride to 4. This is to utilize the data set fully (we don't skip a
single word) but also avoid any overlap between the batches, since more overlap could lead
to increased overfitting.
    
</div>

## TOKEN EMBEDDING


In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])


In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
print(embedding_layer.weight)
print(embedding_layer(input_ids))



## POSITIONAL EMBEDDING

<div class="alert alert-block alert-success">

Previously, we focused on very small embedding sizes in this chapter for illustration
purposes. 

We now consider more realistic and useful embedding sizes and encode the input
tokens into a 256-dimensional vector representation. 

This is smaller than what the original
GPT-3 model used (in GPT-3, the embedding size is 12,288 dimensions) but still reasonable
for experimentation. 

Furthermore, we assume that the token IDs were created by the BPE
tokenizer that we implemented earlier, which has a vocabulary size of 50,257:

</div>

In [ ]:


# Initialize the tokenizer for GPT-2
tokenizer = tiktoken.get_encoding("gpt2")

# Get the vocabulary size for the tokenizer
vocab_sizes = tokenizer.n_vocab 
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_sizes, output_dim)

<div class="alert alert-block alert-info">
    
Using the token_embedding_layer above, if we sample data from the data loader, we
embed each token in each batch into a 256-dimensional vector. If we have a batch size of 8
with four tokens each, the result will be an 8 x 4 x 256 tensor.
    
</div>

In [ ]:
max_length = 4

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:\n", token_embeddings.shape)

<div class="alert alert-block alert-success">

For a GPT model's absolute embedding approach, we just need to create another
embedding layer that has the same dimension as the token_embedding_layer:

</div>

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

<div class="alert alert-block alert-info">
    
As shown in the preceding code example, the input to the pos_embeddings is usually a
placeholder vector torch.arange(context_length), which contains a sequence of
numbers 0, 1, ..., up to the maximum input length − 1. 

The context_length is a variable
that represents the supported input size of the LLM. 

Here, we choose it similar to the
maximum length of the input text. 

In practice, input text can be longer than the supported
context length, in which case we have to truncate the text.
    
</div>

<div class="alert alert-block alert-info">
    
As we can see, the positional embedding tensor consists of four 256-dimensional vectors.
We can now add these directly to the token embeddings, where PyTorch will add the 4x256-
dimensional pos_embeddings tensor to each 4x256-dimensional token embedding tensor in
each of the 8 batches:
    
</div>

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)